# 科创基础数据、违约因变量与 LGD 数据合并

合并键统一为 `cst_id + loanacctno`。先合并基础数据与违约因变量，再合并 LGD 字段。

In [ ]:
from pathlib import Path
import pandas as pd

# ── 输入文件：支持 CSV / XLSX / XLS ──
BASE_FILE = 'kechuang_base.csv'          # 第一个 SQL 导出文件：不含违约因变量
RISK_FILE = 'kechuang_risk_snapshot.csv' # 第二个 SQL 导出文件：仅违约因变量
LGD_FILE  = 'loan_lgd_observation.csv'   # 第三个 SQL 导出文件：LGD 所需字段

# ── 输出文件 ──
OUTPUT_FILE = 'kechuang_base_risk_lgd_merged.csv'

# 左连接保证以第一个文件的样本为准。
MERGE_HOW = 'left'
# 推荐保持 one_to_one：任一文件的合并键重复时立即报错，避免合并后行数异常膨胀。
# 若业务确认存在一对多关系，可改为 None，并自行核对下方的重复键统计。
MERGE_VALIDATE = 'one_to_one'


In [ ]:
def read_table(file_path):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f'找不到文件：{path.resolve()}')
    suffix = path.suffix.lower()
    if suffix == '.csv':
        try:
            df = pd.read_csv(path, encoding='utf-8-sig', low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding='gbk', low_memory=False)
    elif suffix in {'.xlsx', '.xls'}:
        df = pd.read_excel(path)
    else:
        raise ValueError(f'不支持的文件格式：{suffix}；仅支持 csv/xlsx/xls')
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def prepare_keys(df, name):
    key_cols = ['cst_id', 'loanacctno']
    missing = [c for c in key_cols if c not in df.columns]
    if missing:
        raise KeyError(f'{name} 缺少合并键：{missing}；实际字段示例：{df.columns.tolist()[:20]}')
    out = df.copy()
    for col in key_cols:
        out[col] = out[col].astype('string').str.strip()
    if out[key_cols].isna().any().any():
        raise ValueError(f'{name} 的 cst_id 或 loanacctno 存在缺失，不能作为合并键。')
    return out

def report_keys(df, name):
    key_cols = ['cst_id', 'loanacctno']
    dup_mask = df.duplicated(key_cols, keep=False)
    print(f'\n{name}：{len(df):,} 行，{df.shape[1]:,} 列，唯一键 {df[key_cols].drop_duplicates().shape[0]:,} 个')
    print(f'  重复键组合数：{df.loc[dup_mask, key_cols].drop_duplicates().shape[0]:,}')
    print(f'  重复键涉及行数：{int(dup_mask.sum()):,}')

base = prepare_keys(read_table(BASE_FILE), '基础数据文件')
risk = prepare_keys(read_table(RISK_FILE), '违约因变量文件')
lgd = prepare_keys(read_table(LGD_FILE), 'LGD 数据文件')

report_keys(base, '基础数据文件')
report_keys(risk, '违约因变量文件')
report_keys(lgd, 'LGD 数据文件')


In [ ]:
KEYS = ['cst_id', 'loanacctno']

def merge_with_report(left, right, right_name):
    overlap_non_keys = sorted((set(left.columns) & set(right.columns)) - set(KEYS))
    if overlap_non_keys:
        raise ValueError(
            f'与 {right_name} 合并时发现同名非键字段：{overlap_non_keys}。'
            '请先明确保留规则或重命名字段，避免静默覆盖。'
        )
    merged = left.merge(
        right, on=KEYS, how=MERGE_HOW, validate=MERGE_VALIDATE, indicator=True
    )
    print(f'\n合并 {right_name} 后：{len(merged):,} 行，{merged.shape[1] - 1:,} 列')
    print(merged['_merge'].value_counts(dropna=False).rename('行数'))
    merged = merged.drop(columns='_merge')
    return merged

# 第一步：基础数据 + 违约因变量
merged_base_risk = merge_with_report(base, risk, '违约因变量文件')

# 第二步：上述结果 + LGD 所需字段
merged_final = merge_with_report(merged_base_risk, lgd, 'LGD 数据文件')

print('\n最终数据预览：')
display(merged_final.head())
print(f'最终行数：{len(merged_final):,}；最终列数：{merged_final.shape[1]:,}')


In [ ]:
output_path = Path(OUTPUT_FILE)
merged_final.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'已保存：{output_path.resolve()}')
